<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания №17


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

В данной версии реализовано простое, сложное и множественное наследование, полиморфизм с перекрытием и перегрузкой методов, а также добавлены новые атрибуты, методы и универсальные generic-классы.

Множественное наследование реализовано через интерфейсы ITrackable (отслеживание) и IInsurable (страхование).

Сложное наследование реализовано через добавление промежуточного класса CourierDelivery, от которого наследуется ExpressDelivery. Иерархия: ShippingOption -> CourierDelivery -> ExpressDelivery.

Простое наследование демонстрируется на классе Pickup.

Базовые и производные классы расширены новыми свойствами (приоритет доставки, комиссия за обработку, разрешена ли хрупкая тара и т.д.) и методами, включая перегрузку методов расчета стоимости.

Добавлен генерический класс DeliveryManager<T>, обеспечивающий безопасное управление коллекциями опций доставки с ограничениями where T : ShippingOption.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) и реализуйте полиморфизм с перекрытием и прегегрузкой методов, а также generic классы

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [ ]:
using System;
using System.Collections.Generic;

public class Order
{
    private int _orderId;
    private double _packageWeight;
    private string _receiverName;
    private decimal _declaredValue;
    private bool _isFragile; // Новое поле: признак хрупкого груза

    public int OrderId
    {
        get { return _orderId; }
        set { _orderId = value > 0 ? value : 1; }
    }

    public double PackageWeight
    {
        get { return _packageWeight; }
        set { _packageWeight = value > 0 ? value : 0.1; }
    }

    public string ReceiverName
    {
        get { return _receiverName; }
        set { _receiverName = string.IsNullOrWhiteSpace(value) ? "Unknown" : value; }
    }

    public decimal DeclaredValue
    {
        get { return _declaredValue; }
        set { _declaredValue = value >= 0 ? value : 0; }
    }

    public bool IsFragile
    {
        get { return _isFragile; }
        set { _isFragile = value; }
    }

    public Order(int orderId, double packageWeight, string receiverName, decimal declaredValue, bool isFragile = false)
    {
        OrderId = orderId;
        PackageWeight = packageWeight;
        ReceiverName = receiverName;
        DeclaredValue = declaredValue;
        IsFragile = isFragile;
    }

    public void Checkout(ShippingOption option)
    {
        Console.WriteLine($"Order #{OrderId} | Receiver: {ReceiverName} | Weight: {PackageWeight}kg | Fragile: {IsFragile}");
        Console.WriteLine(option.GetDeliveryDetails());
        
        if (option.CheckAvailability(this))
        {
            decimal total = option.CalculateCost(this);
            Console.WriteLine($"Total Cost: {total:C}");
        }
        else
        {
            Console.WriteLine("Error: Shipping option is not available for this order.");
        }
    }

    public string GetOrderSummary()
    {
        return $"Order #{OrderId} - {PackageWeight}kg - {DeclaredValue:C}";
    }

    public void UpdateWeight(double newWeight)
    {
        PackageWeight = newWeight;
    }
}

public interface ITrackable
{
    string TrackingNumber { get; set; }
    string GetTrackingStatus();
}

public interface IInsurable
{
    decimal InsuranceRate { get; set; }
    decimal CalculateInsuranceCost(Order order);
}

public class ShippingOption
{
    private int _id;
    private string _name;
    private decimal _baseCost;
    private double _maxWeight;
    private string _description;
    private bool _isActive;
    
    // Новые атрибуты базового класса
    private int _priorityLevel;
    private decimal _handlingFee;
    private bool _requiresSpecialHandling;

    public int Id
    {
        get { return _id; }
        set { _id = value; }
    }

    public string Name
    {
        get { return _name; }
        set { _name = string.IsNullOrWhiteSpace(value) ? "Unknown" : value; }
    }

    public decimal BaseCost
    {
        get { return _baseCost; }
        set { _baseCost = value >= 0 ? value : 0; }
    }

    public double MaxWeight
    {
        get { return _maxWeight; }
        set { _maxWeight = value > 0 ? value : 1000; }
    }

    public string Description
    {
        get { return _description; }
        set { _description = string.IsNullOrWhiteSpace(value) ? "-" : value; }
    }

    public bool IsActive
    {
        get { return _isActive; }
        set { _isActive = value; }
    }

    public int PriorityLevel
    {
        get { return _priorityLevel; }
        set { _priorityLevel = value > 0 ? value : 1; }
    }

    public decimal HandlingFee
    {
        get { return _handlingFee; }
        set { _handlingFee = value >= 0 ? value : 0; }
    }

    public bool RequiresSpecialHandling
    {
        get { return _requiresSpecialHandling; }
        set { _requiresSpecialHandling = value; }
    }

    public ShippingOption(int id, string name, decimal baseCost, double maxWeight)
    {
        Id = id;
        Name = name;
        BaseCost = baseCost;
        MaxWeight = maxWeight;
        IsActive = true;
        Description = "Standard shipping option";
        PriorityLevel = 1;
        HandlingFee = 50m;
        RequiresSpecialHandling = false;
    }

    // Полиморфный метод расчета стоимости
    public virtual decimal CalculateCost(Order order)
    {
        decimal cost = BaseCost + (order.PackageWeight > 1.0 ? (decimal)(order.PackageWeight - 1.0) * 50m : 0m);
        if (order.IsFragile && RequiresSpecialHandling)
        {
            cost += HandlingFee;
        }
        return cost;
    }

    // ПЕРЕГРУЗКА МЕТОДОВ: расчет стоимости с дополнительным коэффициентом или скидкой
    public virtual decimal CalculateCost(Order order, decimal customMarkup)
    {
        return CalculateCost(order) + customMarkup;
    }

    // ПЕРЕГРУЗКА МЕТОДОВ: расчет стоимости с учетом срочности в днях
    public virtual decimal CalculateCost(Order order, int expeditedDays)
    {
        decimal baseCalc = CalculateCost(order);
        return expeditedDays < 3 ? baseCalc * 1.2m : baseCalc;
    }

    public virtual string GetDeliveryDetails()
    {
        return $"[{Id}] {Name} | Base: {BaseCost:C} | Max Weight: {MaxWeight}kg | Priority: {PriorityLevel}";
    }

    public bool CheckAvailability(Order order)
    {
        return IsActive && order.PackageWeight <= MaxWeight;
    }

    public void ApplyDiscount(decimal percent)
    {
        if (percent > 0 && percent <= 100)
        {
            BaseCost -= BaseCost * (percent / 100);
        }
    }

    public void ToggleActiveStatus()
    {
        IsActive = !IsActive;
    }

    // Новые методы базового класса
    public void SetPriority(int priority)
    {
        PriorityLevel = priority;
    }

    public string GetOptionSummary()
    {
        return $"Option: {Name}, Active: {IsActive}, Priority: {PriorityLevel}";
    }
}

public class CourierDelivery : ShippingOption
{
    private string _courierName;
    private string _vehicleType;
    private string _contactPhone;

    // Новые атрибуты производного класса
    private double _fuelConsumptionRate;
    private bool _isContactlessAllowed;
    private int _deliveryRating;

    public string CourierName
    {
        get { return _courierName; }
        set { _courierName = string.IsNullOrWhiteSpace(value) ? "Pending" : value; }
    }

    public string VehicleType
    {
        get { return _vehicleType; }
        set { _vehicleType = string.IsNullOrWhiteSpace(value) ? "Van" : value; }
    }

    public string ContactPhone
    {
        get { return _contactPhone; }
        set { _contactPhone = string.IsNullOrWhiteSpace(value) ? "N/A" : value; }
    }

    public double FuelConsumptionRate
    {
        get { return _fuelConsumptionRate; }
        set { _fuelConsumptionRate = value > 0 ? value : 8.5; }
    }

    public bool IsContactlessAllowed
    {
        get { return _isContactlessAllowed; }
        set { _isContactlessAllowed = value; }
    }

    public int DeliveryRating
    {
        get { return _deliveryRating; }
        set { _deliveryRating = (value >= 1 && value <= 5) ? value : 5; }
    }

    public CourierDelivery(int id, string name, decimal baseCost, double maxWeight) 
        : base(id, name, baseCost, maxWeight)
    {
        CourierName = "Not assigned";
        VehicleType = "Van";
        ContactPhone = "000-000";
        FuelConsumptionRate = 10.0;
        IsContactlessAllowed = true;
        DeliveryRating = 5;
    }

    public void AssignCourier(string name, string phone)
    {
        CourierName = name;
        ContactPhone = phone;
    }

    public string GetCourierInfo()
    {
        return $"Courier: {CourierName} | Phone: {ContactPhone} | Vehicle: {VehicleType} | Rating: {DeliveryRating}/5";
    }

    public void UpgradeVehicle(string newVehicle)
    {
        VehicleType = newVehicle;
    }

    // Новые методы класса CourierDelivery
    public void SetRating(int rating)
    {
        DeliveryRating = rating;
    }

    public void ToggleContactless()
    {
        IsContactlessAllowed = !IsContactlessAllowed;
    }
}

public class ExpressDelivery : CourierDelivery, ITrackable, IInsurable
{
    private int _minTime;
    private decimal _surcharge;
    private string _trackingNumber;
    private decimal _insuranceRate;

    // Новые атрибуты
    private bool _isJetTransportUsed;
    private int _guaranteedHours;
    private decimal _expressTax;

    public int MinTime
    {
        get { return _minTime; }
        set { _minTime = value > 0 ? value : 1; }
    }

    public decimal Surcharge
    {
        get { return _surcharge; }
        set { _surcharge = value >= 0 ? value : 0; }
    }

    public string TrackingNumber
    {
        get { return _trackingNumber; }
        set { _trackingNumber = string.IsNullOrWhiteSpace(value) ? "TBA" : value; }
    }

    public decimal InsuranceRate
    {
        get { return _insuranceRate; }
        set { _insuranceRate = value >= 0 ? value : 0.01m; }
    }

    public bool IsJetTransportUsed
    {
        get { return _isJetTransportUsed; }
        set { _isJetTransportUsed = value; }
    }

    public int GuaranteedHours
    {
        get { return _guaranteedHours; }
        set { _guaranteedHours = value > 0 ? value : 24; }
    }

    public decimal ExpressTax
    {
        get { return _expressTax; }
        set { _expressTax = value >= 0 ? value : 100m; }
    }

    public ExpressDelivery(int id, string name, decimal baseCost, double maxWeight, int minTime, decimal surcharge) 
        : base(id, name, baseCost, maxWeight)
    {
        MinTime = minTime;
        Surcharge = surcharge;
        TrackingNumber = "TBA";
        InsuranceRate = 0.05m;
        IsJetTransportUsed = false;
        GuaranteedHours = 12;
        ExpressTax = 150m;
        RequiresSpecialHandling = true;
    }

    public override decimal CalculateCost(Order order)
    {
        decimal cost = base.CalculateCost(order) + Surcharge + CalculateInsuranceCost(order);
        if (IsJetTransportUsed) cost += ExpressTax;
        return cost;
    }

    public string GetTrackingStatus()
    {
        return $"[TRK: {TrackingNumber}] Status: In Transit (Express Delivery, Guaranteed: {GuaranteedHours}h)";
    }

    public decimal CalculateInsuranceCost(Order order)
    {
        return order.DeclaredValue * InsuranceRate;
    }

    public void ExpediteProcessing()
    {
        if (MinTime > 1) MinTime -= 1;
    }

    // Новые методы класса ExpressDelivery
    public void EnableJetTransport()
    {
        IsJetTransportUsed = true;
    }

    public string GetExpressSummary()
    {
        return $"Express option with {GuaranteedHours}h delivery window. Jet: {IsJetTransportUsed}";
    }
}

public class Pickup : ShippingOption, ITrackable
{
    private string _address;
    private int _storageDays;
    private string _trackingNumber;

    // Новые атрибуты
    private bool _isCellLockerAvailable;
    private string _workingHours;
    private int _securityCode;

    public string Address
    {
        get { return _address; }
        set { _address = string.IsNullOrWhiteSpace(value) ? "Main Hub" : value; }
    }

    public int StorageDays
    {
        get { return _storageDays; }
        set { _storageDays = value > 0 ? value : 3; }
    }

    public string TrackingNumber
    {
        get { return _trackingNumber; }
        set { _trackingNumber = string.IsNullOrWhiteSpace(value) ? "TBA" : value; }
    }

    public bool IsCellLockerAvailable
    {
        get { return _isCellLockerAvailable; }
        set { _isCellLockerAvailable = value; }
    }

    public string WorkingHours
    {
        get { return _workingHours; }
        set { _workingHours = string.IsNullOrWhiteSpace(value) ? "08:00 - 20:00" : value; }
    }

    public int SecurityCode
    {
        get { return _securityCode; }
        set { _securityCode = value >= 1000 ? value : 1234; }
    }

    public Pickup(int id, string name, decimal baseCost, double maxWeight, string address) 
        : base(id, name, baseCost, maxWeight)
    {
        Address = address;
        StorageDays = 5;
        TrackingNumber = "TBA";
        IsCellLockerAvailable = true;
        WorkingHours = "09:00 - 21:00";
        SecurityCode = 7777;
    }

    public override decimal CalculateCost(Order order)
    {
        return BaseCost;
    }

    public string GetTrackingStatus()
    {
        return $"[TRK: {TrackingNumber}] Status: Ready for Pickup at {Address} (Hours: {WorkingHours})";
    }

    public void ExtendStorage(int days)
    {
        if (days > 0) StorageDays += days;
    }

    public string PrintPickupLabel()
    {
        return $"--- PICKUP LABEL ---\nDeliver to: {Address}\nValid for: {StorageDays} days\nLocker Code: {SecurityCode}";
    }

    // Новые методы класса Pickup
    public void SetWorkingHours(string hours)
    {
        WorkingHours = hours;
    }

    public void RegenerateSecurityCode(int newCode)
    {
        SecurityCode = newCode;
    }
}

// ГЕНЕРИЧЕСКИЙ КЛАСС: Управление коллекцией опций доставки с ограничениями по типу
public class DeliveryManager<T> where T : ShippingOption
{
    private List<T> _options = new List<T>();

    public void AddOption(T option)
    {
        _options.Add(option);
    }

    public void RemoveOption(T option)
    {
        _options.Remove(option);
    }

    public void ExecuteAllCalculations(Order order)
    {
        Console.WriteLine($"\n--- Processing Generic Delivery Manager for Order #{order.OrderId} ---");
        foreach (var option in _options)
        {
            if (option.CheckAvailability(order))
            {
                decimal cost = option.CalculateCost(order);
                Console.WriteLine($"Option '{option.Name}' cost: {cost:C}");
            }
        }
    }
}

public class Program
{
    public static void Main()
    {
        Order order1 = new Order(1001, 2.5, "Ivan Ivanov", 5000m, true);
        Order order2 = new Order(1002, 0.5, "Anna Smith", 1000m, false);
        Order order3 = new Order(1003, 50.0, "Peter Parker", 10000m, false);

        ExpressDelivery express = new ExpressDelivery(10, "Super Express", 500m, 10.0, 12, 200m);
        express.AssignCourier("Alex", "555-1234");
        express.TrackingNumber = "EXP999888";
        express.UpgradeVehicle("Motorcycle");
        express.EnableJetTransport();

        Pickup pickup = new Pickup(20, "Local Post", 100m, 20.0, "Lenina st. 1");
        pickup.TrackingNumber = "PKP111222";
        pickup.ExtendStorage(2);
        pickup.ApplyDiscount(10);
        pickup.SetWorkingHours("08:00 - 22:00");

        order1.Checkout(express);
        Console.WriteLine(express.GetCourierInfo());
        Console.WriteLine(express.GetTrackingStatus());

        // Демонстрация перегруженного метода расчета стоимости
        decimal overloadCost = express.CalculateCost(order1, 150m);
        Console.WriteLine($"Cost with custom markup: {overloadCost:C}");

        Console.WriteLine("\n-------------------\n");

        order2.Checkout(pickup);
        Console.WriteLine(pickup.PrintPickupLabel());
        Console.WriteLine(pickup.GetTrackingStatus());

        Console.WriteLine("\n-------------------\n");

        order3.Checkout(express);

        // Демонстрация генерического класса DeliveryManager<T>
        DeliveryManager<ShippingOption> manager = new DeliveryManager<ShippingOption>();
        manager.AddOption(express);
        manager.AddOption(pickup);

        manager.ExecuteAllCalculations(order1);
    }
}